In [ ]:
def build_full_hypothesis_summary(
    hypothesis_text,
    part_a_text,
    part_a_summary,
    part_b_text,
    part_c_text
):
    """
    Combine Part A, Part B, and Part C into one clinician-readable summary block.
    """

    final_text = f"""
============================================

🌟Final Summary

============================================

⭐Hypothesis: {hypothesis_text}

============================================

⚫In Part A…

Raw Part A report:
{part_a_text}
{part_a_summary}
============================================

⚫In Part B…

{part_b_text}
============================================

⚫In Part C…

{part_c_text}
============================================
""".strip()

    return final_text

In [ ]:
def clean_hypothesis_text(r):
    left = f"{r['level1']} {r['var1']}" if r["level1"] is not None else r["var1"]
    right = f"{r['level2']} {r['var2']}" if r["level2"] is not None else r["var2"]
    return f"Does {left} AND {right} affect your likelihood of Type 2 Diabetes?"

In [ ]:
n = min(len(results), len(PartA_conclusion_results), len(all_final_outputs), len(pipeline_results))

for i in range(n):
    r = results[i]

    hypothesis_text = clean_hypothesis_text(r)
    part_a_text = r["report"]
    part_a_summary = PartA_conclusion_results[i]["conclusion"]
    part_b_text = all_final_outputs[i]["final_output"]
    part_c_text = pipeline_results[i]["result"]

    full_summary = build_full_hypothesis_summary(
        hypothesis_text=hypothesis_text,
        part_a_text=part_a_text,
        part_a_summary=part_a_summary,
        part_b_text=part_b_text,
        part_c_text=part_c_text
    )

    print("\n" + "=" * 80)
    print(full_summary)

# Saving to an excel file

In [ ]:
i = 0

r = results[i]

hypothesis_text = clean_hypothesis_text(r)

part_a_text = r["report"]
part_a_summary = PartA_conclusion_results[i]["conclusion"]
part_b_text = all_final_outputs[i]["final_output"]   # or executor_runs[i]["final_markdown"]
part_c_text = pipeline_results[i]["result"]

full_summary = build_full_hypothesis_summary(
    hypothesis_text=hypothesis_text,
    part_a_text=part_a_text,
    part_a_summary=part_a_summary,
    part_b_text=part_b_text,
    part_c_text=part_c_text
)

print("\n" + "=" * 80)
print(full_summary)

In [ ]:
import pandas as pd

final_summary_rows = []

n = min(
    len(results),
    len(PartA_conclusion_results),
    len(all_final_outputs),
    len(pipeline_results)
)

for i in range(n):

    r = results[i]

    hypothesis_text = clean_hypothesis_text(r)

    part_a_text = r["report"]
    part_a_summary = PartA_conclusion_results[i]["conclusion"]
    part_b_text = all_final_outputs[i]["final_output"]
    part_c_text = pipeline_results[i]["result"]

    full_summary = build_full_hypothesis_summary(
        hypothesis_text=hypothesis_text,
        part_a_text=part_a_text,
        part_a_summary=part_a_summary,
        part_b_text=part_b_text,
        part_c_text=part_c_text
    )

    print("\n" + "=" * 80)
    print(full_summary)

    final_summary_rows.append({
        "question_num": i + 1,
        "hypothesis": hypothesis_text,
        "part_a_summary": part_a_summary,
        "part_b_summary": part_b_text,
        "part_c_summary": part_c_text,
        "full_summary": full_summary
    })

In [ ]:
df_summary = pd.DataFrame(final_summary_rows)

In [ ]:
pd.DataFrame({
    "full_summary":[row["full_summary"] for row in final_summary_rows]
}).to_csv("t2dm_full_summaries.csv", index=False)

pd.DataFrame(final_summary_rows).to_csv(
    "t2dm_hypothesis_summaries.csv",
    index=False
)

df_summary.to_csv("t2dm_hypothesis_summaries.csv", index=False)

# Overall Outcome

In [ ]:
from openai import OpenAI

client = OpenAI()

def build_full_hypothesis_summary(
    hypothesis_text,
    part_a_text,
    part_a_summary,
    part_b_text,
    part_c_text,
    model="gpt-4o-mini"
):
    """
    Builds a final synthesis summary combining results from Part A, B, and C agents.
    """

    prompt = f"""
You are a scientific synthesis agent.

Your job is to combine the findings from three analysis agents
into ONE clear overall conclusion sentence.

The sentence should:
• Begin with "Overall, across the three agents..."
• Be 25–40 words
• Mention whether the evidence is consistent, mixed, weak, or strong
• Refer to the hypothesis
• Be written in a neutral scientific tone

Hypothesis:
{hypothesis_text}

Part A conclusion:
{part_a_summary}

Part B conclusion:
{part_b_text}

Part C conclusion:
{part_c_text}

Write ONE final synthesis sentence.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    final_sentence = response.choices[0].message.content.strip()

    return final_sentence
    

In [ ]:
import pandas as pd

final_summary_rows = []

n = min(
    len(results),
    len(PartA_conclusion_results),
    len(all_final_outputs),
    len(pipeline_results)
)

for i in range(n):
    r = results[i]

    hypothesis_text = clean_hypothesis_text(r)

    part_a_text = r["report"]
    part_a_summary = PartA_conclusion_results[i]["conclusion"]
    part_b_text = all_final_outputs[i]["final_output"]
    part_c_text = pipeline_results[i]["result"]

    full_summary = build_full_hypothesis_summary(
        hypothesis_text=hypothesis_text,
        part_a_text=part_a_text,
        part_a_summary=part_a_summary,
        part_b_text=part_b_text,
        part_c_text=part_c_text
    )

    print("\n" + "=" * 80)
    print(full_summary)

    final_summary_rows.append({
        "question_num": i + 1,
        "hypothesis": hypothesis_text,
        "part_a_summary": part_a_summary,
        "part_b_summary": part_b_text,
        "part_c_summary": part_c_text,
        "full_summary": full_summary
    })

final_summary_df = pd.DataFrame(final_summary_rows)
print(final_summary_df.shape)